# Gitarchey Spark worker (Colab)
Run cells top to bottom. The worker connects **out** to your backend over WSS, so Colab needs no public address.

Before you start:
1. Backend env has `WORKER_TOKEN` (>=32 chars) and is reachable over HTTPS (Render URL, or a tunnel: `cloudflared tunnel --url http://localhost:8000`, then add that hostname to `ALLOWED_HOSTS`).
2. Colab sidebar -> key icon -> **Secrets**: add `GITARCHEY_WORKER_TOKEN` and `GITARCHEY_API_HOST` (host only, e.g. `your-api.onrender.com`), enable *Notebook access* for both.

In [ ]:
# 1. Java 17 + PySpark + websockets
!apt-get -qq update && apt-get -qq install -y openjdk-17-jdk-headless > /dev/null
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]
!java -version
!pip -q install "pyspark==3.5.*" "numpy>=1.26" "websockets>=13,<18"

In [ ]:
# 2. Upload worker.py and tasks.py (from backend/colab_worker/)
from google.colab import files
files.upload()

In [ ]:
# 3. Load secrets + sanity-check the backend is reachable
from google.colab import userdata
import urllib.request
HOST  = userdata.get("GITARCHEY_API_HOST").strip().removeprefix("https://").rstrip("/")
TOKEN = userdata.get("GITARCHEY_WORKER_TOKEN")
URL   = f"wss://{HOST}/api/v1/workers/spark/connect"
try:
    print("backend HTTP:", urllib.request.urlopen(f"https://{HOST}/", timeout=60).status)  # first hit may wake a sleeping Render dyno
except Exception as e:
    print("HTTP check:", e, "(a 404 here is fine - it still means the host is reachable)")

In [ ]:
# 4. Offline self-test of the Spark tasks (no backend needed)
from worker import make_spark
from tasks import TASKS
spark = make_spark()
commits = [{"sha": f"s{i}", "repo_id": "r1", "author": "ann" if i % 3 else "bob", "date": f"2025-01-0{1 + i % 5}T1{i % 10}:00:00+00:00",
            "message": "fix bug in parser" if i % 4 == 0 else "add feature", "additions": 5 + i, "deletions": i % 4,
            "files_changed": 1 + i % 3, "extensions": {".py": 1 + i % 2}} for i in range(30)]
print(TASKS["commit_activity_rdd"](spark, {"group_by": "author"}, {"commits": commits}, lambda p, m: None))

In [ ]:
# 5. Start the worker (background thread). Look for: "connected as colab-1"
from worker import start_in_background
w = start_in_background(URL, TOKEN, worker_id="colab-1")

Keep this tab open (Colab idles out). Submit a job from the frontend Analytics page, or run the optional cell below.
`w.stop()` stops the worker; if the runtime is recycled, re-run cells 1-3 and 5.

In [ ]:
# 6. (optional) submit a job yourself and wait for the result
import requests, time, getpass
API = f"https://{HOST}/api/v1"
email, pw = input("email: "), getpass.getpass("password: ")
tok = requests.post(f"{API}/auth/login", json={"email": email, "password": pw}).json()["access_token"]
H = {"Authorization": f"Bearer {tok}"}
repo_id = requests.get(f"{API}/repositories", headers=H).json()[0]["id"]
job = requests.post(f"{API}/analytics/jobs", headers=H, json={
    "task_type": "commit_activity_rdd", "params": {"group_by": "author"}, "repo_ids": [repo_id]}).json()
while (j := requests.get(f"{API}/analytics/jobs/{job['id']}", headers=H).json())["status"] not in ("succeeded", "failed", "cancelled"):
    print(j["status"], j.get("progress")); time.sleep(3)
print(j["status"], requests.get(f"{API}/analytics/jobs/{job['id']}/results", headers=H).json())